# 01 — OAT (One-At-A-Time ablation)

Reference 1점에서 한 축씩만 갈아끼우며 marginal RMSE 측정. 11축 × 옵션 × 5 seed = 31 cell × 5 = 155 fit.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*.csv`
- **출력**: `4_output/baseline/oat/{master.csv, checkpoint.json}`
- **참조**: [strategy.md §4](strategy.md), [strategy_common.md §6·§7·§8](../strategy_common.md)
- **Resume**: master.csv에 이미 있는 (axis, option, seed) 조합은 자동 skip

## 1. 환경 설정 + 데이터 로드

In [ ]:
import os, sys

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
    PROJECT_ROOT = '/content/project'
except ImportError:
    %run ../../setup.py
    from utils.config import PROJECT_ROOT

BASELINE_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '0_baseline')
if BASELINE_DIR not in sys.path:
    sys.path.insert(0, BASELINE_DIR)

import warnings
warnings.filterwarnings('ignore')

from utils.data import load_all, get_feat_cols, split_xs
import axes

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
axes.set_data(xs, xs_dict, ys, feat_cols)

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

## 2. 출력 경로 + master.csv / checkpoint 로드

- `master.csv` row 단위 영구 박제 (axis/option/seed/oof_rmse/val_rmse/test_rmse/elapsed_sec/timestamp)
- `checkpoint.json` — 완료한 (axis, option, seed) 튜플 set
- 끊김 후 재실행 시 master.csv에 있는 조합은 자동 skip

In [ ]:
# ── 노트북 상단 단일 변수 (strategy_common §8) ──
N_JOBS = 7
N_ESTIMATORS = 100  # strategy.md §11 default

import json
import pandas as pd
from datetime import datetime

OUT_DIR = os.path.join(PROJECT_ROOT, '4_output', 'baseline', 'oat')
os.makedirs(OUT_DIR, exist_ok=True)
MASTER_CSV = os.path.join(OUT_DIR, 'master.csv')
CKPT_JSON  = os.path.join(OUT_DIR, 'checkpoint.json')
META_JSON  = os.path.join(OUT_DIR, 'meta.json')

# cfg 11축은 master row마다 'cfg_<axis>' 컬럼으로 풀어 저장 → 셀 cfg 완전 복원 가능
CFG_COLS = [f'cfg_{ax}' for ax in axes.AXES.keys()]
MASTER_COLS = [
    'axis', 'option', 'seed', 'is_reference',
    'oof_rmse', 'val_rmse', 'test_rmse',
    'elapsed_sec', 'effective_target_transform', 'timestamp',
] + CFG_COLS

# master.csv 로드 (없으면 init)
if os.path.exists(MASTER_CSV):
    master = pd.read_csv(MASTER_CSV)
    done_set = set(zip(master['axis'], master['option'].astype(str), master['seed']))
    print(f'기존 master.csv: {len(master)} rows, done={len(done_set)} (axis,option,seed)')
else:
    master = pd.DataFrame()
    done_set = set()
    print('master.csv 신규 생성')

# meta.json — 재현성용 단일 파일 (run마다 덮어쓰기)
meta = {
    'created':       datetime.now().isoformat(timespec='seconds'),
    'n_jobs':        N_JOBS,
    'n_estimators':  N_ESTIMATORS,
    'reference':     axes.REFERENCE,
    'axes':          {k: list(map(str, v)) for k, v in axes.AXES.items()},
    'seeds':         axes.SEEDS,
    'agg_preset_lib': axes.AGG_PRESET_LIB,
    'tweedie_losses': list(axes.TWEEDIE_LOSSES),  # target_transform 자동 OFF 룰
    'pp_pin': {
        'cleaning': axes.PP_PIN_CLEANING,
        'outlier':  axes.PP_PIN_OUTLIER,
        'binarize': axes.PP_PIN_BINARIZE,
        'iso':      axes.PP_PIN_ISO,
        'lds':      axes.PP_PIN_LDS,
        'ge':       axes.PP_PIN_GE,
    },
    'exclude_cols':  axes.EXCLUDE_COLS,
    'master_cols':   MASTER_COLS,
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str, ensure_ascii=False)
print(f'meta.json saved → {META_JSON}')

## 3. OAT grid 순회 — cell × seed = 155

axes.generate_oat_grid()로 (axis, option, seed, cfg, is_reference) 튜플 생성. 매 fit 완료 시 row append + master.csv 즉시 flush (끊김 보호).

In [ ]:
grid = axes.generate_oat_grid()
todo = [(axis, opt, seed, cfg, is_ref) for (axis, opt, seed, cfg, is_ref) in grid
        if (axis, str(opt), seed) not in done_set]

print(f'전체 grid: {len(grid)} | 완료: {len(grid) - len(todo)} | 남은 cell-seed: {len(todo)}')

In [ ]:
for i, (axis_name, option, seed, cfg, is_ref) in enumerate(todo):
    print(f'\n[{i+1}/{len(todo)}] axis={axis_name} option={option} seed={seed}')
    try:
        result = axes.run_one(
            cfg, seed=seed, n_jobs=N_JOBS, n_estimators=N_ESTIMATORS,
        )
    except Exception as e:
        # strategy_common §13: 에러 숨기지 말고 사용자에게 보고
        print(f'!! ERROR axis={axis_name} option={option} seed={seed}: {e}')
        raise

    row = {
        'axis': axis_name,
        'option': option,
        'seed': seed,
        'is_reference': is_ref,
        'oof_rmse':  result['oof_rmse'],
        'val_rmse':  result['val_rmse'],
        'test_rmse': result['test_rmse'],
        'elapsed_sec': result['elapsed_sec'],
        'effective_target_transform': result['effective_target_transform'],  # tweedie 시 'none' override 추적
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    # cfg 11축 컬럼 풀어 추가 (cfg_CLF, cfg_reg_level, ...) → 셀 cfg 완전 복원
    for ax_name in axes.AXES.keys():
        row[f'cfg_{ax_name}'] = cfg[ax_name]

    master = pd.concat([master, pd.DataFrame([row])], ignore_index=True)
    master[MASTER_COLS].to_csv(MASTER_CSV, index=False)   # 즉시 flush

    done_set.add((axis_name, str(option), seed))
    with open(CKPT_JSON, 'w') as f:
        json.dump({
            'done_count': len(done_set),
            'total': len(grid),
            'last_completed': row,
        }, f, indent=2, default=str)

    print(f'  oof={result["oof_rmse"]:.6f}  val={result["val_rmse"]:.6f}  '
          f'test={result["test_rmse"]:.6f}  elapsed={result["elapsed_sec"]:.1f}s'
          f'  effective_tt={result["effective_target_transform"]}')

print(f'\n[OAT 완료] master.csv: {len(master)} rows')